# DIMENTION MODELING (DIM)

In [0]:
%sql

-- describe history datamodelling.silver.ukhousing

select * from datamodelling.silver.ukhousing
version as of 10

In [0]:
%sql
CREATE OR REPLACE  TABLE datamodelling.gold.dimregion
AS (

WITH region_name AS (
SELECT 
        distinct(region_name)
FROM datamodelling.silver.ukhousing
)
SELECT 
    *,
    row_number() over ( order by Region_Name) as dimregionkey
 FROM region_name
)

In [0]:
%sql
select * from datamodelling.gold.dimregion

In [0]:
%sql
CREATE OR REPLACE TABLE datamodelling.gold.dimareacode
AS(
WITH area_code AS(

SELECT 
    distinct(Area_Code)
 FROM datamodelling.silver.ukhousing
)

SELECT *,
row_number() over (order by Area_Code) as dimareacodekey
FROM area_code
)

In [0]:
%sql
CREATE OR REPLACE TABLE datamodelling.gold.dimdate
AS(
WITH datedim AS (
SELECT 
    distinct(date) as date,
    year,
    month,
    day,
    quarter
FROM datamodelling.silver.ukhousing
)
SELECT 
    *,
    row_number() over (order by date) as dimdatekey
FROM datedim
)

# DIMENTION MODELING (FACT)

In [0]:
%sql

CREATE OR REPLACE TABLE datamodelling.gold.factcashbuyer
AS  (
SELECT  
dimregionkey,
dimareacodekey,
dimdatekey,
Cash_Average_Price,
Cash_Index,
Cash_Monthly_Change,
Cash_Annual_Change,
Cash_Sales_Volume,
row_number() over (order by Cash_Average_Price) as factcashbuyerkey
FROM  datamodelling.silver.ukhousing cash
LEFT JOIN datamodelling.gold.dimregion region
ON cash.region_name = region.region_name

LEFT JOIN datamodelling.gold.dimareacode area
ON cash.area_code = area.Area_Code

LEFT JOIN datamodelling.gold.dimdate date
ON cash.date=date.date

)

In [0]:
%sql
select *  from datamodelling.gold.factcashbuyer

In [0]:
%sql
CREATE OR REPLACE TABLE datamodelling.gold.factmortgagebuyer
AS(
SELECT 
dimregionkey,
dimareacodekey,
dimdatekey,
Mortgage_Average_Price,
Mortgage_Index,
Mortgage_Monthly_Change,
Mortgage_Annual_Change,
Mortgage_Sales_Volume,
row_number() over (order by Mortgage_Average_Price) as factmortgagebuyerkey
FROM  datamodelling.silver.ukhousing mortgage
LEFT JOIN datamodelling.gold.dimregion region
ON mortgage.region_name = region.region_name

LEFT JOIN datamodelling.gold.dimareacode area
ON mortgage.area_code = area.Area_Code

LEFT JOIN datamodelling.gold.dimdate date
ON mortgage.date=date.date
)

# DIMENTION MODELING (QUERY)

In [0]:
%sql
-- select * from datamodelling.silver.ukhousing
-- where Cash_Average_Price=58405
select * from datamodelling.gold.factmortgagebuyer